<a href="https://colab.research.google.com/github/M-AyyanAsif/NETSOL_TASKS/blob/main/unsup_worksheet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Unsupervised Learning: K-Means, DBSCAN, PCA (Practice Sheet)
**NAVTTC / NIAI AI/ML Trainer Cohort.** Needs `numpy`, `scikit-learn` and `matplotlib`. **No internet, no keys.** All data is **synthetic** (made up by us), and the same data as in the slides, so your numbers should match the pictures. Fill every `None` / `# YOUR CODE` area, then run top to bottom. Check cells print `OK` or `FIX`.

In [9]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.datasets import make_moons
from sklearn.preprocessing import StandardScaler
def check(name, ok, hint=""):
    print(("OK   " if ok else "FIX  ") + name + ("" if ok else "  -> " + hint))

# Same customers as in the slides: 45 customers, 2 features
rng = np.random.RandomState(11)
def blob(c, s, n): return rng.normal(c, s, (n, 2))
X = np.vstack([blob((4, 12), (1.4, 1.2), 15), blob((12, 6), (1.8, 1.1), 15), blob((21, 3), (1.8, 0.8), 15)]).clip(0.3)
m = len(X)
print("customers:", m, "| columns: average basket (PKR '000), visits per month")
print("first customer x(1) =", X[0].round(1))

customers: 45 | columns: average basket (PKR '000), visits per month
first customer x(1) = [ 6.4 11.7]


## Task 1 (K-MEANS): Assign by hand

Our starting stars are `mu = [[7.1, 11.4], [16.2, 8.4], [6.9, 9.3]]`. Write `assign(X, mu)` that returns, for every customer, the **number of the closest star** (0, 1 or 2 here; the slides call them 1, 2, 3). Use the Euclidean distance. Then print customer 10, `X[9]`, with its star number.

In [10]:
mu = np.array([[7.1, 11.4], [16.2, 8.4], [6.9, 9.3]])
def assign(X, mu):
    # Calculate pairwise Euclidean distance and find the index of the minimum distance for each row
    return np.argmin(np.linalg.norm(X[:, np.newaxis] - mu, axis=2), axis=1)
c = assign(X, mu)

In [11]:
try:
    check("one number per customer", len(c) == m, "return an array of length m")
    check("customer 10 joins star 0 (slide 5: c = 1)", int(c[9]) == 0, "use argmin over the distances to each star")
    print("customer 10 =", X[9].round(1), "-> star", int(c[9]))
except Exception as e:
    print('FIX  this task is not finished yet (' + type(e).__name__ + '). Fill in the YOUR CODE area above and run it again.')

OK   one number per customer
OK   customer 10 joins star 0 (slide 5: c = 1)
customer 10 = [ 4.  11.2] -> star 0


## Task 2 (K-MEANS): Move the stars and compute J

Write `move(X, c, K)` returning the new stars (the **mean** of the customers assigned to each star), and `cost(X, c, mu)` returning **J = the average squared distance** from each customer to its own star. Then run assign and move five times and print J each round: it should never go up.

In [12]:
def move(X, c, K):
    return np.array([X[c == k].mean(axis=0) if np.any(c == k) else np.zeros(X.shape[1]) for k in range(K)])

def cost(X, c, mu):
    # Calculate the squared distance for each point to its assigned star, and return the mean
    return np.mean(np.sum((X - mu[c]) ** 2, axis=1))

mu0 = np.array([[7.1, 11.4], [16.2, 8.4], [6.9, 9.3]])
mu = mu0.copy(); Js = []
for t in range(5):
    c = assign(X, mu)
    Js.append(cost(X, c, mu))
    mu = move(X, c, 3)
print([None if j is None else round(j, 1) for j in Js])

[np.float64(28.6), np.float64(15.6), np.float64(14.2), np.float64(5.3), np.float64(3.3)]


In [13]:
try:
    check("J never goes up", all(Js[i+1] <= Js[i] + 1e-9 for i in range(len(Js)-1)), "check move() uses the mean of the assigned customers")
    check("first J is about 28.6", abs(Js[0] - 28.6) < 0.2, "J = average of squared distances, not the sum or the square root")
    check("last J is about 3.3", abs(Js[-1] - 3.3) < 0.2, "five rounds are enough on this data")
except Exception as e:
    print('FIX  this task is not finished yet (' + type(e).__name__ + '). Fill in the YOUR CODE area above and run it again.')

OK   J never goes up
OK   first J is about 28.6
OK   last J is about 3.3


## Task 3 (K-MEANS): Elbow and lucky starts

With scikit-learn: for K = 1 to 8 fit `KMeans(K, n_init=10, random_state=0)` and keep `inertia_ / m` (this is J). Then fit **one** random start per seed (`init="random", n_init=1, random_state=seed`) for seeds 0 to 59 and print the **best and worst J**.

In [14]:
Ks = list(range(1, 9))
Jk = []
for K in Ks:
    kmeans = KMeans(n_clusters=K, n_init=10, random_state=0)
    kmeans.fit(X)
    Jk.append(kmeans.inertia_ / m)

seed_J = []
for seed in range(60):
    kmeans_rand = KMeans(n_clusters=3, init="random", n_init=1, random_state=seed)
    kmeans_rand.fit(X)
    seed_J.append(kmeans_rand.inertia_ / m)

print([round(j, 1) for j in Jk])

[62.4, 17.2, 3.3, 2.3, 1.9, 1.4, 1.3, 1.1]


In [15]:
try:
    check("8 values of J", len(Jk) == 8, "one J for every K from 1 to 8")
    check("elbow: big drop from K=2 to K=3", Jk[1] - Jk[2] > 10 * (Jk[2] - Jk[3]), "check your loop")
    check("60 seeds", len(seed_J) == 60, "one J for each seed 0..59")
    print("best J =", round(min(seed_J), 1), " worst J =", round(max(seed_J), 1))
    check("some starts are unlucky", max(seed_J) > 3 * min(seed_J), "use init='random' and n_init=1")
except Exception as e:
    print('FIX  this task is not finished yet (' + type(e).__name__ + '). Fill in the YOUR CODE area above and run it again.')

OK   8 values of J
OK   elbow: big drop from K=2 to K=3
OK   60 seeds
best J = 3.3  worst J = 16.5
OK   some starts are unlucky


## Task 4 (DBSCAN): Curved groups and odd customers

**Part A.** Make `make_moons(300, noise=0.06, random_state=3)`. Fit `KMeans(2)` and `DBSCAN(eps=0.2, min_samples=5)` and count the clusters DBSCAN finds (ignore the noise label -1).
**Part B.** Add five odd customers to our data, scale everything, and run `DBSCAN(eps=0.45, min_samples=4)`. Count the dots labelled `-1` (noise).

In [16]:
Xm, _ = make_moons(300, noise=0.06, random_state=3)
km_labels = KMeans(2, random_state=0).fit_predict(Xm)
db_model = DBSCAN(eps=0.2, min_samples=5).fit(Xm)
db_labels = db_model.labels_
n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)

odd = np.array([[25, 12], [26, 14], [1, 1], [27, 9], [14, 15]])
Xf = np.vstack([X, odd])
Xs = StandardScaler().fit_transform(Xf)
noise_model = DBSCAN(eps=0.45, min_samples=4).fit(Xs)
noise_labels = noise_model.labels_
n_noise = np.sum(noise_labels == -1)

In [17]:
try:
    check("DBSCAN finds 2 moons", n_clusters == 2, "eps=0.2, min_samples=5")
    check("all five planted customers are noise", all(noise_labels[45:] == -1), "scale first, then eps=0.45")
    print("noise dots:", n_noise, "(5 planted + the rest are normal customers flagged by mistake)")
    check("a flag is not a verdict: noise count is 6", n_noise == 6, "expected 5 planted + 1 normal edge customer")
except Exception as e:
    print('FIX  this task is not finished yet (' + type(e).__name__ + '). Fill in the YOUR CODE area above and run it again.')

OK   DBSCAN finds 2 moons
OK   all five planted customers are noise
noise dots: 6 (5 planted + the rest are normal customers flagged by mistake)
OK   a flag is not a verdict: noise count is 6


## Task 5 (PCA): Keep 90%

Make 10 features from 3 hidden factors (code given). **Scale** the data, fit `PCA()` and compute the **cumulative** explained variance. Then find the smallest number of components that keeps at least 90% and transform the data with that many.

In [18]:
r4 = np.random.RandomState(9)
Z = r4.normal(0, 1, (300, 3)); Wt = r4.normal(0, 1, (3, 10))
Xp = Z @ Wt + r4.normal(0, 0.5, (300, 10))
Xp_s = StandardScaler().fit_transform(Xp)
pca = PCA().fit(Xp_s)
cum = np.cumsum(pca.explained_variance_ratio_)
k90 = np.argmax(cum >= 0.90) + 1
X_small = PCA(n_components=k90).fit_transform(Xp_s)

In [19]:
try:
    check("3 components keep at least 90%", k90 == 3, "use np.cumsum and find the first value >= 0.90")
    check("91% kept with 3", abs(cum[2] - 0.907) < 0.01, "scale before PCA")
    check("data is now 300 x 3", X_small.shape == (300, 3), "PCA(n_components=k90).fit_transform")
    print("kept:", [round(v, 2) for v in cum[:4]])
except Exception as e:
    print('FIX  this task is not finished yet (' + type(e).__name__ + '). Fill in the YOUR CODE area above and run it again.')

OK   3 components keep at least 90%
OK   91% kept with 3
OK   data is now 300 x 3
kept: [np.float64(0.41), np.float64(0.73), np.float64(0.91), np.float64(0.94)]


## Reflect (answer in two lines each)
1. K-Means found 3 groups here. What would you check before telling a manager 'we have 3 customer types'?
2. DBSCAN flagged one normal customer. What would you do with a flag in a real fraud team?
3. PCA gave you 3 numbers instead of 10. What did you gain and what did you lose?